In [ ]:
import numpy
import torch
import torchvision
import time
import random

def manual_conv2d(x, weight, stride=1, padding=0):
    N, C_in, H_in, W_in = x.shape
    C_out, _, K, _ = weight.shape # 需要注意的是 C_out 和 C_in 是不同的
    H_out = (H_in + 2 * padding - K) // stride + 1
    W_out = (W_in + 2 * padding - K) // stride + 1
    x_unfolded = torch.nn.functional.unfold(x, kernel_size=K, stride=stride, padding=padding)
    w_reshaped = weight.view(C_out, -1)
    # out_unfolded = torch.einsum('cd, ndl -> ncl', w_reshaped, x_unfolded)
    x_permuted = x_unfolded.permute(0, 2, 1)          # (N, L, D)  D = C_in*K*K
    temp = torch.matmul(x_permuted, w_reshaped.T)     # (N, L, C_out)
    out_unfolded = temp.permute(0, 2, 1)              # (N, C_out, L)
    out = out_unfolded.view(N, C_out, H_out, W_out)
    return out

def manual_avg_pool2d(x, kernel_size, stride=None, padding=0):
    N, C, H, W = x.shape
    if stride is None:
        stride = kernel_size
    # 计算输出尺寸
    H_out = (H + 2 * padding - kernel_size) // stride + 1
    W_out = (W + 2 * padding - kernel_size) // stride + 1
    # 对输入进行 padding
    if padding > 0:
        x = torch.nn.functional.pad(x, (padding, padding, padding, padding), mode='constant', value=0)
    # 使用 unfold 提取滑动窗口
    x_unfolded = torch.nn.functional.unfold(x, kernel_size, stride=stride, padding=0)
    # x_unfolded shape: (N, C*K*K, L)
    L = x_unfolded.size(2)
    # 变形为 (N, C, K*K, L) 并求平均
    x_unfolded = x_unfolded.view(N, C, kernel_size * kernel_size, L)
    out_unfolded = x_unfolded.mean(dim=2)  # (N, C, L)
    # 变形回图像形状
    out = out_unfolded.view(N, C, H_out, W_out)
    return out

def create_conv2d_param(in_channels, out_channels, kernel_size, device=None):
    weight = torch.empty(out_channels, in_channels, kernel_size, kernel_size, device=device)
    torch.nn.init.kaiming_normal_(weight, mode='fan_out', nonlinearity='relu')
    return torch.nn.Parameter(weight)

def manual_batch_norm(x, running_mean, running_var, weight, bias, training=True, momentum=0.1):
    N, C, H, W = x.shape
    if training:
        mean = x.mean(dim=(0, 2, 3))
        var = x.var(dim=(0, 2, 3), unbiased=False)
        with torch.no_grad():
            running_mean.copy_(momentum * mean + (1 - momentum) * running_mean)
            running_var.copy_(momentum * var + (1 - momentum) * running_var)
    else:
        mean = running_mean
        var = running_var
    mean = mean.view(1, C, 1, 1)
    var = var.view(1, C, 1, 1)
    weight = weight.view(1, C, 1, 1)
    bias = bias.view(1, C, 1, 1)
    x_normalized = (x - mean) * torch.rsqrt(var + 1e-05)
    out = weight * x_normalized + bias
    return out

def create_bn_param(num_features, device=None):
    weight = torch.nn.Parameter(torch.ones(num_features, device=device))
    bias = torch.nn.Parameter(torch.zeros(num_features, device=device))
    running_mean = torch.zeros(num_features, device=device)
    running_var = torch.ones(num_features, device=device)
    return weight, bias, running_mean, running_var

def create_linear_param(in_features, out_features, device=None):
    weight = torch.empty(out_features, in_features, device=device)
    torch.nn.init.kaiming_uniform_(weight, a=5 ** 0.5)
    bias = torch.nn.Parameter(torch.zeros(out_features, device=device))
    return torch.nn.Parameter(weight), bias

def residual_block(x, layer_name, block_idx, stride, weights, is_training):
    """
    残差模块的前向计算
    x: 输入张量
    layer_name: 层名，如 'layer1', 'layer2'
    block_idx: 块索引，0 或 1
    stride: 卷积步长
    weights: 包含所有参数的字典
    is_training: 是否为训练模式（bool）
    """
    identity = x

    # Conv1
    out = manual_conv2d(x, weights[f'{layer_name}_b{block_idx}_conv1_w'], stride=stride, padding=1)
    out = manual_batch_norm(out,
                            weights[f'{layer_name}_b{block_idx}_bn1_rm'],
                            weights[f'{layer_name}_b{block_idx}_bn1_rv'],
                            weights[f'{layer_name}_b{block_idx}_bn1_w'],
                            weights[f'{layer_name}_b{block_idx}_bn1_b'],
                            training=is_training)
    out = torch.nn.functional.relu(out)

    # Conv2
    out = manual_conv2d(out, weights[f'{layer_name}_b{block_idx}_conv2_w'], stride=1, padding=1)
    out = manual_batch_norm(out,
                            weights[f'{layer_name}_b{block_idx}_bn2_rm'],
                            weights[f'{layer_name}_b{block_idx}_bn2_rv'],
                            weights[f'{layer_name}_b{block_idx}_bn2_w'],
                            weights[f'{layer_name}_b{block_idx}_bn2_b'],
                            training=is_training)

    # Shortcut
    if f'{layer_name}_b{block_idx}_sc_w' in weights:
        identity = manual_conv2d(identity, weights[f'{layer_name}_b{block_idx}_sc_w'], stride=stride, padding=0)
        identity = manual_batch_norm(identity,
                                     weights[f'{layer_name}_b{block_idx}_sc_bn_rm'],
                                     weights[f'{layer_name}_b{block_idx}_sc_bn_rv'],
                                     weights[f'{layer_name}_b{block_idx}_sc_bn_w'],
                                     weights[f'{layer_name}_b{block_idx}_sc_bn_b'],
                                     training=is_training)

    out += identity
    out = torch.nn.functional.relu(out)
    return out


def resnet18(x, weights, is_training):
    """
    ResNet-18 前向计算
    x: 输入张量
    weights: 包含所有参数的字典
    is_training: 是否为训练模式（bool）
    """
    out = manual_conv2d(x, weights['conv1_w'], stride=1, padding=1)
    out = manual_batch_norm(out,
                            weights['bn1_rm'], weights['bn1_rv'],
                            weights['bn1_w'], weights['bn1_b'],
                            training=is_training)
    out = torch.nn.functional.relu(out)

    # Layer 1
    out = residual_block(out, 'layer1', 0, 1, weights, is_training)
    out = residual_block(out, 'layer1', 1, 1, weights, is_training)

    # Layer 2
    out = residual_block(out, 'layer2', 0, 2, weights, is_training)
    out = residual_block(out, 'layer2', 1, 1, weights, is_training)

    # Layer 3
    out = residual_block(out, 'layer3', 0, 2, weights, is_training)
    out = residual_block(out, 'layer3', 1, 1, weights, is_training)

    # Layer 4
    out = residual_block(out, 'layer4', 0, 2, weights, is_training)
    out = residual_block(out, 'layer4', 1, 1, weights, is_training)

    out = manual_avg_pool2d(out, 4)
    out = out.view(out.size(0), -1)
    out = torch.nn.functional.linear(out, weights['fc_w'], weights['fc_b'])

    return out

def train_preprocess(img_tensor, mean, std):
    img = torchvision.transforms.functional.pad(img_tensor, padding=4, fill=0)
    top = random.randint(0, 8)
    left = random.randint(0, 8)
    img = torchvision.transforms.functional.crop(img, top, left, 32, 32)

    if random.random() < 0.5:
        img = torchvision.transforms.functional.hflip(img)

    img = img.float() / 255.0
    img = (img - mean.view(3, 1, 1)) / std.view(3, 1, 1)
    return img

def test_preprocess(img_tensor, mean, std):
    img = img_tensor.float() / 255.0
    img = (img - mean.view(3, 1, 1)) / std.view(3, 1, 1)
    return img

def initialize_residual_block_weights(weights, layer_name, block_idx, in_ch, out_ch, has_shortcut, device):
    """
    向 weights 字典中添加一个残差块的所有参数
    """
    # Conv1
    weights[f'{layer_name}_b{block_idx}_conv1_w'] = create_conv2d_param(in_ch, out_ch, 3, device=device)
    w, b, rm, rv = create_bn_param(out_ch, device=device)
    weights[f'{layer_name}_b{block_idx}_bn1_w'], weights[f'{layer_name}_b{block_idx}_bn1_b'] = w, b
    weights[f'{layer_name}_b{block_idx}_bn1_rm'], weights[f'{layer_name}_b{block_idx}_bn1_rv'] = rm, rv

    # Conv2
    weights[f'{layer_name}_b{block_idx}_conv2_w'] = create_conv2d_param(out_ch, out_ch, 3, device=device)
    w, b, rm, rv = create_bn_param(out_ch, device=device)
    weights[f'{layer_name}_b{block_idx}_bn2_w'], weights[f'{layer_name}_b{block_idx}_bn2_b'] = w, b
    weights[f'{layer_name}_b{block_idx}_bn2_rm'], weights[f'{layer_name}_b{block_idx}_bn2_rv'] = rm, rv

    # Shortcut (if needed)
    if has_shortcut:
        weights[f'{layer_name}_b{block_idx}_sc_w'] = create_conv2d_param(in_ch, out_ch, 1, device=device)
        w, b, rm, rv = create_bn_param(out_ch, device=device)
        weights[f'{layer_name}_b{block_idx}_sc_bn_w'], weights[f'{layer_name}_b{block_idx}_sc_bn_b'] = w, b
        weights[f'{layer_name}_b{block_idx}_sc_bn_rm'], weights[f'{layer_name}_b{block_idx}_sc_bn_rv'] = rm, rv

def initialize_weights(device):
    weights = {}

    weights['conv1_w'] = create_conv2d_param(3, 64, 3, device=device)
    w, b, rm, rv = create_bn_param(64, device=device)
    weights['bn1_w'], weights['bn1_b'] = w, b
    weights['bn1_rm'], weights['bn1_rv'] = rm, rv

    # Layer1 (64 channels, both blocks stride=1, no shortcut)
    initialize_residual_block_weights(weights, 'layer1', 0, 64, 64, False, device)
    initialize_residual_block_weights(weights, 'layer1', 1, 64, 64, False, device)

    # Layer2 (first block: 64->128, stride=2, shortcut; second: 128->128, stride=1, no shortcut)
    initialize_residual_block_weights(weights, 'layer2', 0, 64, 128, True, device)
    initialize_residual_block_weights(weights, 'layer2', 1, 128, 128, False, device)

    # Layer3 (first block: 128->256, stride=2, shortcut; second: 256->256, stride=1)
    initialize_residual_block_weights(weights, 'layer3', 0, 128, 256, True, device)
    initialize_residual_block_weights(weights, 'layer3', 1, 256, 256, False, device)

    # Layer4 (first block: 256->512, stride=2, shortcut; second: 512->512, stride=1)
    initialize_residual_block_weights(weights, 'layer4', 0, 256, 512, True, device)
    initialize_residual_block_weights(weights, 'layer4', 1, 512, 512, False, device)

    # Fully connected layer
    fc_w, fc_b = create_linear_param(512, 10, device=device)
    weights['fc_w'], weights['fc_b'] = fc_w, fc_b

    return weights

def train(weights, optimizer, train_data, train_labels, batch_size, device, mean, std):
    weights = {k: v for k, v in weights.items()}  # ensure we work on the dict
    num_samples = len(train_labels)
    indices = list(range(num_samples))
    random.shuffle(indices)

    total_loss = 0.0
    total_correct = 0
    total_batches = 0

    for i in range(0, num_samples, batch_size):
        batch_indices = indices[i:i+batch_size]
        batch_images = train_data[batch_indices]          # raw uint8 images, shape (B, C, H, W)
        batch_labels = train_labels[batch_indices]

        # 正则化
        preprocessed = []
        for img in batch_images:
            # img is a uint8 tensor on device, shape (C, H, W)
            img = train_preprocess(img, mean, std)       # uses random crop & flip
            preprocessed.append(img)
        batch_x = torch.stack(preprocessed, dim=0).to(device)

        logits = resnet18(batch_x, weights, is_training=True)
        loss = torch.nn.functional.cross_entropy(logits, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_indices)
        total_correct += (torch.argmax(logits, dim=1) == batch_labels).sum().item()
        total_batches += 1

    avg_loss = total_loss / num_samples
    accuracy = total_correct / num_samples * 100.0
    return avg_loss, accuracy

def test(weights, test_data, test_labels, batch_size, device, mean, std):
    weights = {k: v for k, v in weights.items()}
    num_samples = len(test_labels)
    total_correct = 0

    for i in range(0, num_samples, batch_size):
        batch_images = test_data[i:i+batch_size]
        batch_labels = test_labels[i:i+batch_size]

        # Apply test preprocessing (only normalisation, no random transforms)
        preprocessed = []
        for img in batch_images:
            img = test_preprocess(img, mean, std)
            preprocessed.append(img)
        batch_x = torch.stack(preprocessed, dim=0).to(device)

        with torch.no_grad():
            logits = resnet18(batch_x, weights, is_training=False)
        total_correct += (torch.argmax(logits, dim=1) == batch_labels).sum().item()

    accuracy = total_correct / num_samples * 100.0
    return accuracy

if __name__ == '__main__':
    EPOCHS = 100
    BATCH_SIZE = 128
    LEARNING_RATE = 0.01
    TRAIN_ACC_THRESHOLD = 90.0
    SGD_MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4

    train_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=True, download=True, transform=None)
    test_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=False, download=True, transform=None)
    train_data_np = train_set.data.transpose(0, 3, 1, 2)
    test_data_np = test_set.data.transpose(0, 3, 1, 2)

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")

    train_data = torch.tensor(train_data_np, device=device)
    train_labels = torch.tensor(train_set.targets, dtype=torch.long, device=device)
    test_data = torch.tensor(test_data_np, device=device)
    test_labels = torch.tensor(test_set.targets, dtype=torch.long, device=device)

    data = train_set.data.astype(numpy.float64) / 255.0
    mean = torch.tensor(data.mean(axis=(0, 1, 2)), dtype=torch.float32).to(device)
    std = torch.tensor(data.std(axis=(0, 1, 2)), dtype=torch.float32).to(device)

    weights = initialize_weights(device)
    trainable_params = []
    
    # running_mean 和 running_var 不是 torch.nn.Parameter，我们要把它俩拿掉
    for name, param in weights.items():
        if isinstance(param, torch.nn.Parameter):
            trainable_params.append(param)

    optimizer = torch.optim.SGD(trainable_params, lr=LEARNING_RATE,
                          momentum=SGD_MOMENTUM, weight_decay=WEIGHT_DECAY)

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        train_loss, train_acc = train(weights, optimizer,
                                      train_data, train_labels,
                                      BATCH_SIZE, device, mean, std)
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch:2d}: TrainTime = {epoch_time:.2f}s, "
              f"TrainLoss = {train_loss:.4f}, TrainAcc = {train_acc:.2f}%", end=' ')

        if train_acc >= TRAIN_ACC_THRESHOLD:
            test_acc = test(weights, test_data, test_labels,
                            BATCH_SIZE, device, mean, std)
            print(f"TestAcc = {test_acc:.2f}%")
        else:
            print()

Using device: cuda
Epoch  1: TrainTime = 117.19s, TrainLoss = 1.5815, TrainAcc = 41.15% 
Epoch  2: TrainTime = 118.18s, TrainLoss = 1.1304, TrainAcc = 59.12% 
Epoch  3: TrainTime = 118.79s, TrainLoss = 0.8981, TrainAcc = 68.08% 
Epoch  4: TrainTime = 118.79s, TrainLoss = 0.7430, TrainAcc = 73.81% 
Epoch  5: TrainTime = 118.76s, TrainLoss = 0.6418, TrainAcc = 77.57% 
Epoch  6: TrainTime = 118.80s, TrainLoss = 0.5791, TrainAcc = 79.87% 
Epoch  7: TrainTime = 118.78s, TrainLoss = 0.5208, TrainAcc = 81.87% 
Epoch  8: TrainTime = 118.78s, TrainLoss = 0.4750, TrainAcc = 83.49% 
Epoch  9: TrainTime = 118.75s, TrainLoss = 0.4417, TrainAcc = 84.63% 
Epoch 10: TrainTime = 118.76s, TrainLoss = 0.4074, TrainAcc = 85.76% 
Epoch 11: TrainTime = 118.73s, TrainLoss = 0.3734, TrainAcc = 87.06% 
Epoch 12: TrainTime = 118.73s, TrainLoss = 0.3535, TrainAcc = 87.79% 
Epoch 13: TrainTime = 118.73s, TrainLoss = 0.3291, TrainAcc = 88.35% 
Epoch 14: TrainTime = 118.76s, TrainLoss = 0.3111, TrainAcc = 89.08% 
E

```bash
(base) root@VM-0-80-ubuntu:/workspace/cifar10# nvidia-smi
Fri Apr  3 09:49:37 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:09.0 Off |                    0 |
| N/A   65C    P0    69W /  70W |   5246MiB / 15360MiB |     93%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                                  |
|  GPU   GI   CI        PID   Type   Process name                  GPU Memory |
|        ID   ID                                                   Usage      |
|=============================================================================|
+-----------------------------------------------------------------------------+
```